# Lecture 25 - Introduction to Machine Learning with Scikit-Learn

## Learning Objectives

- Distinguish between supervised and unsupervised learning
- Understand the difference between classification and regression
- Use train_test_split() to create train and test sets
- Fit a model with .fit() and predict with .predict()
- Build LinearRegression, KNeighborsClassifier, and LogisticRegression models

## Key Topics

- Supervised vs unsupervised learning
- Classification vs regression
- train_test_split()
- Feature matrices (X) and target vectors (y)
- model.fit(X, y) and model.predict(X_test)
- LinearRegression, KNeighborsClassifier, LogisticRegression

## Machine Learning Taxonomy

Machine learning algorithms are broadly divided into two camps: **supervised** and **unsupervised** learning.

In **supervised learning**, we have a dataset with input features and known target labels. The model learns to map from inputs to outputs. Think of it as learning with a teacher — the correct answers are provided during training. Common tasks include **classification** (predicting a category — spam vs not spam) and **regression** (predicting a continuous value — house price).

In **unsupervised learning**, we have only input features and no target labels. The model must find hidden structure in the data on its own — clustering customers into segments or reducing dimensionality for visualisation are typical examples.

Scikit-learn provides a consistent API for all these tasks. Every estimator follows the same pattern: import the class, instantiate it, call `.fit()` to learn from data, and then call `.predict()` or `.transform()` to apply the model to new data.

In [ ]:
# Toy example: create synthetic classification data
from sklearn.datasets import make_classification
import pandas as pd

X, y = make_classification(n_samples=200, n_features=4, n_informative=3,
                           n_redundant=0, random_state=42)
df = pd.DataFrame(X, columns=[f"feat_{i}" for i in range(4)])
df["target"] = y
print(df.shape)
print(df["target"].value_counts())

In [ ]:
# Train / test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape[0]}  Test size: {X_test.shape[0]}")
print(f"Class proportions in train:\n{pd.Series(y_train).value_counts(normalize=True)}")

## Feature Matrices (X) and Target Vectors (y)

Scikit-learn expects data in a very specific shape: **X** must be a 2D array-like (matrix) where each row is an observation and each column is a feature. **y** must be a 1D array-like (vector) containing the target value for each observation.

This design is deliberate — it forces you to separate your predictors from your target before modelling, which clarifies the modelling task. Pandas DataFrames (for X) and Series (for y) work natively with scikit-learn, making the transition from data wrangling to modelling seamless.

In [ ]:
# Explicitly separate features and target using Pandas
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="species")

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print(X.head())

## Fitting a Model and Making Predictions

The scikit-learn API follows a two-step workflow:

1. **Fit**: `model.fit(X_train, y_train)` — the model learns parameters from the training data.
2. **Predict**: `model.predict(X_test)` — the model produces predictions on unseen test data.

For classification, you can also use `.predict_proba()` to get class probabilities instead of hard labels. This is especially useful when you want to calibrate confidence thresholds or compute ROC curves.

Never evaluate a model on data it was trained on; that would give an overly optimistic estimate of performance. Always hold out a test set.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print("First 10 predictions:", y_pred[:10])
print("First 10 true labels:", y_test[:10])
print(f"Accuracy: {(y_pred == y_test).mean():.3f}")

In [ ]:
# predict_proba gives probability estimates
probs = knn.predict_proba(X_test)
print("Class probabilities for first 5 test samples:")
print(probs[:5].round(3))

## First Models: LinearRegression, KNeighborsClassifier, LogisticRegression

Three workhorse models every data scientist should know:

- **LinearRegression** — fits a straight line (or hyperplane) through the data. Simple, interpretable, and fast. Best for relationships that are roughly linear.
- **KNeighborsClassifier** — predicts the class of a point by majority vote among its k nearest neighbours. Non-parametric and flexible, but sensitive to the scale of features and the choice of k.
- **LogisticRegression** — despite its name, a classification model. It models the log-odds of class membership as a linear combination of features. Provides well-calibrated probabilities and is a strong baseline for binary and multiclass problems.

Each of these models is a great starting point for any new dataset.

In [ ]:
# LinearRegression on synthetic regression data
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
import numpy as np

X_reg, y_reg = make_regression(n_samples=100, n_features=1, noise=15, random_state=42)
lr = LinearRegression()
lr.fit(X_reg, y_reg)
y_pred_reg = lr.predict(X_reg)

print(f"Coefficient: {lr.coef_[0]:.3f}")
print(f"Intercept: {lr.intercept_:.3f}")

In [ ]:
# LogisticRegression on the Iris dataset
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris

iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

logreg = LogisticRegression(max_iter=200)
logreg.fit(X_tr, y_tr)
print(f"Logistic Regression accuracy: {logreg.score(X_te, y_te):.3f}")
print("Coefficient shape:", logreg.coef_.shape)

In [ ]:
# KNN classification on Iris dataset
knn_iris = KNeighborsClassifier(n_neighbors=3)
knn_iris.fit(X_tr, y_tr)
y_pred_iris = knn_iris.predict(X_te)
print(f"KNN (k=3) accuracy: {(y_pred_iris == y_te).mean():.3f}")

# Try different k values
for k in [1, 3, 5, 10, 20]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr, y_tr)
    acc = knn.score(X_te, y_te)
    print(f"  k={k:2d}  accuracy={acc:.3f}")

## Data Science Connection

Understanding the train/test split and scikit-learn's .fit() / .predict() API is the foundation of every machine learning pipeline you will build. From predicting house prices to classifying medical images, this same pattern applies. The models introduced here — LinearRegression, KNeighborsClassifier, and LogisticRegression — are the building blocks you will pit against more complex models in later lectures to establish baselines and gauge improvement.